 # CLV Prediction — Two-Stage Hybrid Model
 **Pipeline:**
 1. Setup & Configuration
 2. Data Loading + Preprocessing
 3. Feature Engineering
 4. Skip-Gram Customer Embeddings
 5. Sequence Array Generation
 6. S2S Encoder-Decoder Architecture (definition only)
 7. GBM Regression — Data Splits & DMatrix Preparation
 8. Classifier — Data Splits & DMatrix Preparation
 9. Pipeline Execution — Train all 4 models, then evaluate

 ## 1. Setup & Configuration

In [ ]:
# Uncomment to install dependencies in a fresh environment:
# !pip install xgboost tensorflow keras pandas numpy scipy scikit-learn openpyxl h5py requests

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack as sp_vstack
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers

warnings.filterwarnings("ignore")

In [ ]:
# Paths
DATA_PATH  = "https://raw.githubusercontent.com/TheoRusu/customer-lifetime-value/refs/heads/main/Online%20Retail.csv"
CACHE_PATH = "."
DATASET    = "UKRetail"

# Time splits (daily)
MAX_TIME    = 258
TARGET_LEN  = 28                            # 4 weeks
MIN_VALID   = MAX_TIME - 2 * TARGET_LEN + 1 # 203
MAX_VALID   = MAX_TIME - TARGET_LEN         # 230
MIN_TEST    = MAX_TIME - TARGET_LEN + 1     # 231
MAX_TEST    = MAX_TIME                      # 258
SEQ_IN_LEN  = 2 * TARGET_LEN               # 56  – encoder input window
SEQ_OUT_LEN = TARGET_LEN                   # 28  – decoder output window
MIN_TRAIN   = 35
MAX_TRAIN   = MIN_VALID - 1                 # 202

# Feature names
SEQ_BASE_FEATS = [
    "sumProfit", "numberOfUniqueItems",
    "timeSinceLastPurchase", "timeSinceFirstPurchase", "cumSumPurchases",
]
EMBED_FEAT = "CustomerID"

ADD_FEAT_NAMES = [
    "totalNumberOfItems", "totalProfit", "firstRecentGapBetweenOrders",
    "meanOrderProfit", "minItemPrice", "diffMaxMinOrderProfit",
    "varRelativeNumberOfRepurchases", "totalNumberOfRecords",
    "daysSinceLastOrder", "meanItemPrice", "CustomerCountry",
]
FEATS_TO_SCALE = [
    "numberOfOrders", "totalProfit", "meanOrderProfit",
    "totalNumberOfItems", "totalNumberOfRecords",
]

EXCL_NOT_IN_TRAIN = True

# Skip-gram embeddings
SG_EMBED_DIM = 64
SG_WINDOW    = 11
SG_NEG       = 1
SG_EPOCHS    = 50
USE_SG_CACHE = True

# S2S model
ENC_UNITS   = 50
DEC_UNITS   = 50
DROPOUT     = 0.2
LR          = 1e-4
N_EPOCHS    = 50
BATCH_SIZE  = 128
PATIENCE    = 3
STOP_FACTOR = 10
STOP_DELAY  = 5

# GBM
SEED    = 1
NROUNDS = 500
ETA     = 0.05
DEPTH   = 8
SUB     = 0.8
COL_SMP = 0.8
EARLY   = 20

 ## 2. Data Loading + Preprocessing

In [ ]:
ORDER_CACHE = os.path.join(CACHE_PATH, f"customer_order_dt_{DATASET}.pkl")

if os.path.exists(ORDER_CACHE):
    order_df = pd.read_pickle(ORDER_CACHE)
    print(f"Loaded cached order data: {len(order_df):,} rows")
else:
    raw = pd.read_csv(DATA_PATH, encoding="latin-1")
    raw.columns = [
        "OrderID", "ItemID", "ItemName", "ItemQuantity",
        "OrderDate", "ItemPrice", "CustomerID", "CustomerCountry",
    ]

    raw["OrderDate"]  = pd.to_datetime(raw["OrderDate"])
    raw["ItemProfit"] = raw["ItemQuantity"] * raw["ItemPrice"]

    # Drop rows without a customer
    raw = raw.dropna(subset=["CustomerID"]).copy()

    # Encode categoricals as 1-indexed integers
    for col in ["CustomerID", "ItemID", "OrderID", "CustomerCountry"]:
        raw[col] = pd.Categorical(raw[col]).codes + 1

    # ItemSubcategory: first two words of ItemName
    raw["ItemSubcategory"] = (
        raw["ItemName"].fillna("").str.split().str[:2].str.join(" ")
    )
    raw["ItemSubcategory"] = pd.Categorical(raw["ItemSubcategory"]).codes + 1

    # Outlier filtering: remove top-20% and bottom-1% customers by total profit
    cust_profit = raw.groupby("CustomerID")["ItemProfit"].sum()
    excl_custs  = cust_profit[
        (cust_profit >= cust_profit.quantile(0.80)) |
        (cust_profit <= cust_profit.quantile(0.01))
    ].index
    ip_lo = raw["ItemProfit"].quantile(0.01)
    ip_hi = raw["ItemProfit"].quantile(0.80)

    order_df = raw[
        (raw["ItemProfit"] >= ip_lo) &
        (raw["ItemProfit"] <= ip_hi) &
        (~raw["CustomerID"].isin(excl_custs))
    ].copy()

    # Add DayID and WeekID
    min_date = order_df["OrderDate"].min()
    order_df["DayID"]  = (order_df["OrderDate"] - min_date).dt.days.astype(int)
    order_df["WeekID"] = order_df["DayID"] // 7
    
    order_df = order_df[order_df["DayID"] <= MAX_TIME].copy()

    order_df = (
        order_df
        .sort_values(["CustomerID", "OrderDate", "ItemID"])
        .reset_index(drop=True)
    )
    order_df.to_pickle(ORDER_CACHE)
    print(f"Preprocessed and cached: {len(order_df):,} rows")

print(f"Customers: {order_df['CustomerID'].nunique():,}  "
      f"Days: {order_df['DayID'].min()}-{order_df['DayID'].max()}")

 ## 3. Feature Engineering

 ### 3a. Time-Series Panel
 Build a dense (customer × day) panel with sequential features used by the S2S model.

In [ ]:
TS_CACHE = os.path.join(CACHE_PATH, f"ts_panel_{DATASET}.pkl")

if os.path.exists(TS_CACHE):
    ts_df = pd.read_pickle(TS_CACHE)
    print(f"Loaded cached time-series panel: {len(ts_df):,} rows")
else:
    # Daily aggregates
    daily = (
        order_df
        .groupby(["CustomerID", "DayID"])
        .agg(
            sumProfit=("ItemProfit", "sum"),
            numberOfUniqueItems=("ItemID", "nunique"),
        )
        .reset_index()
    )

    # Complete (customer × day) panel, fill missing days with 0
    all_custs = sorted(daily["CustomerID"].unique())
    all_days  = np.arange(order_df["DayID"].min(), order_df["DayID"].max() + 1)
    full_idx  = pd.MultiIndex.from_product(
        [all_custs, all_days], names=["CustomerID", "DayID"]
    )
    ts_df = (
        daily.set_index(["CustomerID", "DayID"])
        .reindex(full_idx, fill_value=0)
        .reset_index()
    )

    # Sequential features (computed per-customer over time)
    BASE_VAL = 10_000

    def add_ts_feats(g):
        day          = g["DayID"].values.astype(float)
        has_purchase = g["sumProfit"].values != 0

        # timeSinceLastPurchase
        last_day    = np.where(has_purchase, day, -BASE_VAL)
        cummax_last = np.maximum.accumulate(last_day)
        tslp        = day - cummax_last

        # timeSinceFirstPurchase
        first_day    = np.where(has_purchase, day, BASE_VAL)
        cummin_first = np.minimum.accumulate(first_day)
        tsfp         = day - cummin_first
        tsfp         = np.where(tsfp < 0, 2 * BASE_VAL + tsfp, tsfp)

        # cumulative purchase count
        cum_purch = np.cumsum(has_purchase)

        g = g.copy()
        g["timeSinceLastPurchase"]  = tslp
        g["timeSinceFirstPurchase"] = tsfp
        g["cumSumPurchases"]        = cum_purch
        return g

    _cust_ids = ts_df["CustomerID"].copy()
    ts_df = ts_df.groupby("CustomerID", group_keys=False).apply(add_ts_feats)
    if "CustomerID" not in ts_df.columns:
        ts_df["CustomerID"] = _cust_ids
    ts_df = ts_df.sort_values(["CustomerID", "DayID"]).reset_index(drop=True)
    ts_df.to_pickle(TS_CACHE)
    print(f"Computed and cached time-series panel: {len(ts_df):,} rows")

 ### 3b. Snapshot Feature Engineering

 For every snapshot time T (T = 0 … MAX_TIME), compute static customer features
 from all transactions with DayID ≤ T.

In [ ]:
FEAT_CACHE   = os.path.join(CACHE_PATH, f"cust_feats_all_{DATASET}.pkl")
TARGET_CACHE = os.path.join(CACHE_PATH, f"target_all_{DATASET}.pkl")
SUBCAT_CACHE = os.path.join(CACHE_PATH, f"subcat_sparse_{DATASET}.pkl")

N_SUBCATEGORIES = int(order_df["ItemSubcategory"].max())


def compute_snapshot_features(order_sub, snapshot_time, target_time_length,
                               all_subcategories):
    if len(order_sub) == 0:
        return None, None, None

    order_sub = order_sub.copy()
    max_date  = order_sub["OrderDate"].max()
    order_sub["DaysSinceOrder"] = (max_date - order_sub["OrderDate"]).dt.days

    # Order-level aggregation
    order_lvl = (
        order_sub
        .groupby(["CustomerID", "OrderDate"])
        .agg(
            DaysSinceOrder=("DaysSinceOrder", "min"),
            OrderProfit=("ItemProfit", "sum"),
            nOrders=("OrderID", "nunique"),
        )
        .reset_index()
    )

    def _order_agg(g):
        days    = np.sort(g["DaysSinceOrder"].values)
        profits = g["OrderProfit"].values
        gaps    = np.diff(days)

        return pd.Series({
            "daysSinceLastOrder":           days[0]               if len(days) >= 1 else np.nan,
            "daysSinceFirstOrder":          days[-1]              if len(days) >= 1 else np.nan,
            "medianDaysSinceOrder":         np.median(days),
            "meanDaysSinceOrder":           days.mean(),
            "maxGapBetweenOrders":          gaps.max()            if len(gaps) >= 1 else np.nan,
            "meanGapBetweenOrders":         gaps.mean()           if len(gaps) >= 1 else np.nan,
            "firstRecentGapBetweenOrders":  gaps[0]               if len(gaps) >= 1 else np.nan,
            "secondRecentGapBetweenOrders": gaps[1]               if len(gaps) >= 2 else np.nan,
            "numberOfOrders":               g["nOrders"].sum(),
            "totalProfit":                  profits.sum(),
            "meanOrderProfit":              profits.mean(),
            "minOrderProfit":               profits.min(),
            "maxOrderProfit":               profits.max(),
            "varDaysSinceOrder":            np.var(days, ddof=1)         if len(days) >= 2 else np.nan,
            "varGapBetweenOrders":          np.var(gaps, ddof=1)         if len(gaps) >= 2 else np.nan,
            "varOrderProfit":               np.var(profits, ddof=1)      if len(profits) >= 2 else np.nan,
            "meanDiffOrderProfit":          np.diff(profits).mean()      if len(profits) >= 2 else np.nan,
            "sdOrderDate":                  g["OrderDate"].map(lambda d: d.toordinal()).std()
                                            if len(g) >= 2 else np.nan,
            "meanOrderDate":                g["OrderDate"].map(lambda d: d.toordinal()).mean(),
            "numberOfOrdersLast3Months":    g.loc[g["DaysSinceOrder"] <= 90, "nOrders"].sum(),
            # lag features (most recent 2 orders)
            "profitOrderLag_0":             profits[np.argmin(days)]     if len(profits) >= 1 else 0.0,
            "profitOrderLag_1":             profits[np.argsort(days)[1]] if len(profits) >= 2 else 0.0,
            "daysSinceOrderLag_0":          days[0]                      if len(days) >= 1 else 0.0,
            "daysSinceOrderLag_1":          days[1]                      if len(days) >= 2 else 0.0,
            "numberOfOrderLag_0":           g["nOrders"].iloc[np.argmin(days)] if len(g) >= 1 else 0,
            "numberOfOrderLag_1":           g["nOrders"].iloc[np.argsort(days)[1]] if len(g) >= 2 else 0,
        })

    cust_order_feats = order_lvl.groupby("CustomerID").apply(_order_agg).reset_index()

    # Derived order features
    cust_order_feats["diffDaysLastFirstOrder"]        = (
        cust_order_feats["daysSinceLastOrder"] - cust_order_feats["daysSinceFirstOrder"])
    cust_order_feats["fracFirstMeanGapBetweenOrders"] = (
        cust_order_feats["firstRecentGapBetweenOrders"] / cust_order_feats["meanGapBetweenOrders"])
    cust_order_feats["fracDaysLastFirstOrder"]        = (
        cust_order_feats["diffDaysLastFirstOrder"] / cust_order_feats["daysSinceLastOrder"])
    cust_order_feats["diffMaxMinOrderProfit"]         = (
        cust_order_feats["maxOrderProfit"] - cust_order_feats["minOrderProfit"])

    # Item-level aggregation
    item_purch = (
        order_sub
        .groupby(["CustomerID", "OrderDate", "ItemID"])
        .agg(ItemQuantity=("ItemQuantity", "sum"), OrderID=("OrderID", "min"))
        .reset_index()
    )
    # Mark first purchase of each item per customer
    min_order_by_cust_item = (
        item_purch.groupby(["CustomerID", "ItemID"])["OrderID"]
        .min().rename("minOrderID").reset_index()
    )
    item_purch = item_purch.merge(min_order_by_cust_item, on=["CustomerID", "ItemID"])
    item_purch["isFirstOrder"] = (item_purch["OrderID"] == item_purch["minOrderID"]).astype(int)

    # Item-level repurchase stats
    item_repurchase = (
        item_purch.groupby("ItemID")
        .apply(lambda g: pd.Series({
            "numberOfOrdersItem":             g["OrderID"].nunique(),
            "numberOfCustomersItem":          g["CustomerID"].nunique(),
            "absoluteNumberOfRepurchases":    g.loc[g["isFirstOrder"] == 0, "OrderID"].nunique(),
            "numberOfCustomersReorderedItem": g.loc[g["isFirstOrder"] == 0, "CustomerID"].nunique(),
        }))
        .reset_index()
    )
    item_repurchase["relativeNumberOfRepurchases"] = (
        item_repurchase["absoluteNumberOfRepurchases"] /
        item_repurchase["numberOfOrdersItem"].replace(0, np.nan)
    )
    item_repurchase["relativeNumberOfCustomersReorderedItem"] = (
        item_repurchase["numberOfCustomersReorderedItem"] /
        item_repurchase["numberOfCustomersItem"].replace(0, np.nan)
    )

    mean_repurchases = (
        item_purch[item_purch["isFirstOrder"] == 0]
        .groupby(["CustomerID", "ItemID"])["OrderID"].nunique()
        .groupby("ItemID").mean()
        .rename("meanNumberOfRepurchases")
        .reset_index()
    )
    item_repurchase = item_repurchase.merge(mean_repurchases, on="ItemID", how="left")
    item_repurchase["meanNumberOfRepurchases"] = item_repurchase["meanNumberOfRepurchases"].fillna(0)

    order_sub_item = order_sub.merge(item_repurchase, on="ItemID", how="left")

    def _item_agg(g):
        return pd.Series({
            "numberOfUniqueItems":                          g["ItemID"].nunique(),
            "numberOfUniqueSubcategories":                  g["ItemSubcategory"].nunique(),
            "minItemPrice":                                 g["ItemPrice"].min(),
            "meanItemPrice":                                g["ItemPrice"].mean(),
            "maxItemPrice":                                 g["ItemPrice"].max(),
            "minRelativeNumberOfRepurchases":               g["relativeNumberOfRepurchases"].min(),
            "maxRelativeNumberOfRepurchases":               g["relativeNumberOfRepurchases"].max(),
            "meanRelativeNumberOfRepurchases":              g["relativeNumberOfRepurchases"].mean(),
            "varRelativeNumberOfRepurchases":               g["relativeNumberOfRepurchases"].var(),
            "minMeanNumberOfRepurchases":                   g["meanNumberOfRepurchases"].min(),
            "maxMeanNumberOfRepurchases":                   g["meanNumberOfRepurchases"].max(),
            "meanMeanNumberOfRepurchases":                  g["meanNumberOfRepurchases"].mean(),
            "varItemPrice":                                 g["ItemPrice"].var(),
            "maxItemPriceDiff":                             g["ItemPrice"].max() - g["ItemPrice"].min(),
            "totalNumberOfItems":                           g["ItemQuantity"].sum(),
            "totalNumberOfRecords":                         len(g),
        })

    cust_item_feats = order_sub_item.groupby("CustomerID").apply(_item_agg).reset_index()

    # CustomerCountry feature
    cust_country = (
        order_sub.groupby("CustomerID")["CustomerCountry"].min().reset_index()
    )

    # Merge all customer features
    feats = (
        cust_order_feats
        .merge(cust_item_feats, on="CustomerID", how="outer")
        .merge(cust_country,    on="CustomerID", how="outer")
    )
    feats["DayID"]  = snapshot_time
    feats = feats.reset_index(drop=True)
    feats["RowID"]  = np.arange(len(feats))

    # Subcategory sparse matrix
    subcat_counts = (
        order_sub
        .groupby(["CustomerID", "ItemSubcategory"])["OrderID"]
        .nunique()
        .rename("count")
        .reset_index()
    )
    subcat_counts = subcat_counts.merge(
        feats[["CustomerID", "RowID"]], on="CustomerID", how="inner"
    )
    rows = subcat_counts["RowID"].values
    cols = (subcat_counts["ItemSubcategory"].values - 1).astype(int)
    vals = subcat_counts["count"].values.astype(float)
    subcat_mat = csr_matrix(
        (vals, (rows, cols)),
        shape=(len(feats), all_subcategories),
    )

    # Target: total profit in the next target_time_length days
    future = order_df[
        (order_df["DayID"] > snapshot_time) &
        (order_df["DayID"] <= snapshot_time + target_time_length)
    ]
    target = (
        future.groupby("CustomerID")["ItemProfit"].sum()
        .rename("CustomerTotalProfit").reset_index()
    )
    target = feats[["CustomerID", "RowID"]].merge(target, on="CustomerID", how="left")
    target["CustomerTotalProfit"] = target["CustomerTotalProfit"].fillna(0)

    return feats, target, subcat_mat


# Run the feature engineering loop (once per snapshot time, cached)

if (os.path.exists(FEAT_CACHE) and
        os.path.exists(TARGET_CACHE) and
        os.path.exists(SUBCAT_CACHE)):
    cust_feats_all = pd.read_pickle(FEAT_CACHE)
    target_all     = pd.read_pickle(TARGET_CACHE)
    with open(SUBCAT_CACHE, "rb") as f:
        subcat_sparse_all = pickle.load(f)
    print(f"Loaded cached features: {len(cust_feats_all):,} rows")
else:
    import gc
    max_day = int(order_df["DayID"].max())

    # Step-7 sample plus critical split days that must always be present
    critical_days = {MAX_TRAIN, MIN_VALID - 1, MIN_TEST - 1}
    all_snapshot_days = (
        set(max_day - s for s in range(0, max_day + 1, 7))
        | {d for d in critical_days if 0 <= d <= max_day}
    )
    shifts_to_run = sorted(max_day - d for d in all_snapshot_days)

    feats_list  = []
    target_list = []
    subcat_list = []

    for i, shift in enumerate(shifts_to_run):
        snapshot_time = max_day - shift
        order_sub = order_df[order_df["DayID"] <= snapshot_time].copy()

        feats, tgt, subcat = compute_snapshot_features(
            order_sub, snapshot_time, TARGET_LEN, N_SUBCATEGORIES
        )
        if feats is None:
            continue

        feats_list.append(feats)
        target_list.append(tgt)
        subcat_list.append(subcat)

        if i % 10 == 0:
            print(f"  snapshot {snapshot_time} ({i+1}/{len(shifts_to_run)})")

    cust_feats_all = pd.concat(feats_list, ignore_index=True)
    target_all     = pd.concat(target_list, ignore_index=True)
    subcat_sparse_all = sp_vstack(subcat_list)

    del feats_list, target_list, subcat_list
    gc.collect()

    # Lag features: totalProfitLag_1 … totalProfitLag_7
    for lag in range(1, 8):
        lag_df = cust_feats_all[["CustomerID", "DayID", "totalProfit"]].copy()
        lag_df["DayID"] += lag
        lag_df = lag_df.rename(columns={"totalProfit": f"totalProfitLag_{lag}"})
        cust_feats_all = cust_feats_all.merge(
            lag_df, on=["CustomerID", "DayID"], how="left"
        )
        cust_feats_all[f"totalProfitLag_{lag}"] = (
            cust_feats_all[f"totalProfitLag_{lag}"].fillna(0)
        )

    cust_feats_all.to_pickle(FEAT_CACHE)
    target_all.to_pickle(TARGET_CACHE)
    with open(SUBCAT_CACHE, "wb") as f:
        pickle.dump(subcat_sparse_all, f)
    print(f"Feature engineering done: {len(cust_feats_all):,} rows")

 ### 3c. Feature Post-Processing
 Scale cumulative features by observation window length, attach past-target features,
 carry forward NaN values, and merge snapshot features into the time-series panel.

In [ ]:
# Scale cumulative features
def scale_feat(val, day_id, min_train, target_len):
    denom = np.maximum(day_id - min_train + 1, target_len)
    return val * target_len / denom

for feat in FEATS_TO_SCALE:
    if feat in cust_feats_all.columns:
        cust_feats_all[feat] = scale_feat(
            cust_feats_all[feat].values,
            cust_feats_all["DayID"].values,
            MIN_TRAIN, TARGET_LEN,
        )

# Past-target features (average profit earned in previous TARGET_LEN window)
past_target = target_all[["CustomerID", "CustomerTotalProfit"]].copy()
past_target["DayID"] = cust_feats_all["DayID"].values
past_target["DayID"] += TARGET_LEN
past_target = past_target[past_target["DayID"] <= MAX_TRAIN]
past_target["CustomerAverageProfitLastTimeLength"] = (
    past_target["CustomerTotalProfit"] / TARGET_LEN
)
past_target = past_target.sort_values(["CustomerID", "DayID"])
past_target["CustomerAverageProfitPastHistory"] = (
    past_target.groupby("CustomerID")["CustomerAverageProfitLastTimeLength"]
    .transform(lambda x: x.expanding().mean())
)
cust_feats_all = cust_feats_all.merge(
    past_target[["CustomerID", "DayID",
                 "CustomerAverageProfitLastTimeLength",
                 "CustomerAverageProfitPastHistory"]],
    on=["CustomerID", "DayID"], how="left",
)
cust_feats_all["CustomerAverageProfitLastTimeLength"] = (
    cust_feats_all["CustomerAverageProfitLastTimeLength"].fillna(0)
)
cust_feats_all["CustomerAverageProfitPastHistory"] = (
    cust_feats_all["CustomerAverageProfitPastHistory"].fillna(0)
)

# Exclude customers with no training history
if EXCL_NOT_IN_TRAIN:
    train_custs   = set(
        order_df[(order_df["DayID"] >= MIN_TRAIN) &
                 (order_df["DayID"] <= MAX_TRAIN)]["CustomerID"]
    )
    non_train_ids = (
        set(order_df[(order_df["DayID"] > MAX_TRAIN) |
                     (order_df["DayID"] < MIN_TRAIN)]["CustomerID"])
        - train_custs
    )
    keep_mask = ~cust_feats_all["CustomerID"].isin(non_train_ids)
    keep_idx  = cust_feats_all[keep_mask].index

    cust_feats_all    = cust_feats_all.loc[keep_idx].reset_index(drop=True)
    target_all        = target_all.loc[keep_idx].reset_index(drop=True)
    subcat_sparse_all = subcat_sparse_all[keep_idx.values]

# Merge snapshot features into time-series panel
feat_cols_for_seq = ["CustomerID", "DayID"] + ADD_FEAT_NAMES
feat_add_df = cust_feats_all[
    [c for c in feat_cols_for_seq if c in cust_feats_all.columns]
].copy()

ts_df = ts_df[~ts_df["CustomerID"].isin(non_train_ids if EXCL_NOT_IN_TRAIN else [])]
ts_df = ts_df.merge(feat_add_df, on=["CustomerID", "DayID"], how="left")

# Forward-fill snapshot features within each customer's time series
for feat in ADD_FEAT_NAMES:
    if feat in ts_df.columns:
        ts_df[feat] = (
            ts_df.sort_values(["CustomerID", "DayID"])
            .groupby("CustomerID")[feat]
            .transform(lambda x: x.ffill())
        )

num_cols = ts_df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    ts_df[col] = ts_df[col].replace([np.inf, -np.inf], np.nan)
    ts_df[col] = ts_df[col].fillna(-20000)

# Final sequence feature list: base features + add features + embedding placeholder
SEQ_FEATS = SEQ_BASE_FEATS + ADD_FEAT_NAMES + [EMBED_FEAT]
print(f"Time-series panel after merge: {len(ts_df):,} rows, "
      f"{len(SEQ_FEATS)} sequence features")

 ## 4. Skip-Gram Customer Embeddings

 Adapts Word2Vec skip-gram to customers: items are "words", customers who
 co-purchased an item are its context.  Produces a 64-dim vector per customer.

In [ ]:
SG_CACHE = os.path.join(CACHE_PATH, f"skipgram_embeddings_{DATASET}_train.pkl")

if USE_SG_CACHE and os.path.exists(SG_CACHE):
    sg_embedding_matrix = pd.read_pickle(SG_CACHE)
    print(f"Loaded cached skip-gram embeddings: {sg_embedding_matrix.shape}")
else:
    from tensorflow.keras.preprocessing.sequence import skipgrams as keras_skipgrams

    # Build item-customer list
    sg_orders = order_df[
        (order_df["DayID"] >= MIN_TRAIN) & (order_df["DayID"] <= MAX_TRAIN)
    ].copy()
    n_customers = int(sg_orders["CustomerID"].max())

    item_cust_map = (
        sg_orders.groupby("ItemID")["CustomerID"]
        .apply(list).to_dict()
    )

    # Build skip-gram dataset
    all_pairs, all_labels = [], []
    for item_id, custs in item_cust_map.items():
        if len(custs) < 2:
            continue
        pairs, labels = keras_skipgrams(
            custs,
            vocabulary_size=n_customers + 1,
            window_size=SG_WINDOW,
            negative_samples=SG_NEG,
            seed=SEED,
        )
        all_pairs.extend(pairs)
        all_labels.extend(labels)

    all_pairs  = np.array(all_pairs,  dtype=np.int32)
    all_labels = np.array(all_labels, dtype=np.float32)

    # Skip-gram Keras model
    target_in  = layers.Input(shape=(1,), name="target")
    context_in = layers.Input(shape=(1,), name="context")

    sg_embed = layers.Embedding(n_customers + 1, SG_EMBED_DIM, name="sg_embed")

    target_vec  = layers.Flatten()(sg_embed(target_in))
    context_vec = layers.Flatten()(sg_embed(context_in))

    dot_prod   = layers.Dot(axes=1)([target_vec, context_vec])
    sg_output  = layers.Dense(1, activation="sigmoid")(dot_prod)

    sg_model = Model([target_in, context_in], sg_output)
    sg_model.compile(loss="binary_crossentropy", optimizer="adam")

    sg_ds = (
        tf.data.Dataset
        .from_tensor_slices(
            ({"target": all_pairs[:, 0], "context": all_pairs[:, 1]}, all_labels)
        )
        .shuffle(100_000)
        .batch(512)
        .repeat()
    )
    steps = max(1, len(all_labels) // 512)
    sg_model.fit(sg_ds, steps_per_epoch=steps, epochs=SG_EPOCHS, verbose=0)

    sg_embedding_matrix = sg_model.get_layer("sg_embed").get_weights()[0]
    pd.to_pickle(sg_embedding_matrix, SG_CACHE)
    print(f"Trained skip-gram embeddings: {sg_embedding_matrix.shape}")

N_EMBED_DIM = sg_embedding_matrix.shape[1]   # 64

# Add customer embedding columns to ts_df
embed_df = pd.DataFrame(
    sg_embedding_matrix,
    columns=[f"embed_{i+1}" for i in range(N_EMBED_DIM)],
)
embed_df.index.name = "CustomerID_idx"
embed_df["CustomerID"] = np.arange(len(embed_df))

def get_embed(cid):
    idx = int(cid)
    if 0 <= idx < len(sg_embedding_matrix):
        return sg_embedding_matrix[idx]
    return np.zeros(N_EMBED_DIM)

embed_cols = [f"embed_{i+1}" for i in range(N_EMBED_DIM)]
embed_wide = pd.DataFrame(
    sg_embedding_matrix,
    columns=embed_cols,
)
embed_wide["CustomerID"] = np.arange(len(embed_wide))

 ## 5. Sequence Array Generation

 Converts the time-series panel into 3-D arrays `(samples, timesteps, features)`
 for the S2S model.  A sliding-window approach generates multiple sequences per
 customer to augment training data.

In [ ]:
def build_sequence_arrays(ts_df, seq_feats, seq_in_len, seq_out_len,
                           min_time, max_time):
    seq_in_plus_out = seq_in_len + seq_out_len

    X_list, Y_list = [], []
    customers = ts_df["CustomerID"].unique()

    for cust in customers:
        cust_ts = ts_df[
            (ts_df["CustomerID"] == cust) &
            (ts_df["DayID"] >= min_time) &
            (ts_df["DayID"] <= max_time)
        ].sort_values("DayID")

        if len(cust_ts) < seq_in_plus_out:
            continue

        mat = cust_ts[seq_feats].values.astype(np.float32)
        n_windows = len(mat) - seq_in_plus_out + 1

        for start in range(n_windows):
            x = mat[start: start + seq_in_len]
            y = mat[start + seq_in_len: start + seq_in_plus_out, 0:1]
            X_list.append(x)
            Y_list.append(y)

    X_arr = np.stack(X_list).astype(np.float32)
    Y_arr = np.stack(Y_list).astype(np.float32)
    return X_arr, Y_arr


def build_validation_array(ts_df, seq_feats, seq_in_len, seq_out_len,
                            val_min_time):
    in_start  = val_min_time - seq_in_len
    in_end    = val_min_time - 1
    out_start = val_min_time
    out_end   = val_min_time + seq_out_len - 1

    X_rows, Y_rows, cust_ids = [], [], []
    for cust, g in ts_df.groupby("CustomerID"):
        g = g.sort_values("DayID")
        x_sub = g[(g["DayID"] >= in_start) & (g["DayID"] <= in_end)]
        y_sub = g[(g["DayID"] >= out_start) & (g["DayID"] <= out_end)]

        if len(x_sub) != seq_in_len or len(y_sub) != seq_out_len:
            continue

        X_rows.append(x_sub[seq_feats].values.astype(np.float32))
        Y_rows.append(y_sub[["sumProfit"]].values.astype(np.float32))
        cust_ids.append(cust)

    X_arr = np.stack(X_rows)
    Y_arr = np.stack(Y_rows)
    return X_arr, Y_arr, np.array(cust_ids)


print("Building training arrays (sliding window)…")
X_train, Y_train = build_sequence_arrays(
    ts_df, SEQ_FEATS, SEQ_IN_LEN, SEQ_OUT_LEN, MIN_TRAIN, MAX_TRAIN
)
print(f"  Train: X={X_train.shape}  Y={Y_train.shape}")

# Buyer-only subset used by the stacking base models (not the standalone S2S)
buyer_seq_mask   = Y_train[:, :, 0].sum(axis=1) > 0
X_train_buyers   = X_train[buyer_seq_mask]
Y_train_buyers   = Y_train[buyer_seq_mask]
print(f"  Train buyers: X={X_train_buyers.shape}  Y={Y_train_buyers.shape}")

print("Building validation arrays…")
X_valid, Y_valid, valid_cust_ids = build_validation_array(
    ts_df, SEQ_FEATS, SEQ_IN_LEN, SEQ_OUT_LEN, MIN_VALID
)
print(f"  Valid: X={X_valid.shape}  Y={Y_valid.shape}")


X_valid2, Y_valid2, valid2_cust_ids = build_validation_array(
    ts_df, SEQ_FEATS, SEQ_IN_LEN, SEQ_OUT_LEN,
    val_min_time=MIN_VALID + SEQ_OUT_LEN
)
print(f"  Valid2 (test): X={X_valid2.shape}  Y={Y_valid2.shape}")

NUM_SEQ_FEATS  = X_train.shape[2]
EMBED_FEAT_IDX = SEQ_FEATS.index(EMBED_FEAT)

 ## 6. S2S Encoder-Decoder Model

 Architecture:
 - Each sequential feature is sliced - 2× causal Conv1D (temporal smoothing)
 - CustomerID - skip-gram embedding (frozen)
 - All features concatenated along the feature axis
 - Encoder GRU 1 reads the full window - hidden state h1
 - Encoder GRU 2 reads the same window, initialized from h1 - hidden state h2
 - Decoder GRU reads the same window, initialized from h2
 - TimeDistributed(Dense(1)) - per-timestep profit prediction
 - Slice last SEQ_OUT_LEN steps as the forecast

In [ ]:
def build_encoder_decoder(seq_in_len, seq_out_len, num_feats,
                           embed_feat_idx, sg_matrix,
                           enc_units=50, dec_units=50, dropout=0.2,
                           lr=1e-4):
    sg_n_rows, sg_dim = sg_matrix.shape

    all_input = layers.Input(shape=(seq_in_len, num_feats), name="seq_input")

    feat_layers = []
    conv_idx = 0
    for i in range(num_feats):
        # Slice feature i
        feat_slice = layers.Lambda(
            lambda x, idx=i: tf.expand_dims(x[:, :, idx], axis=-1),
            name=f"slice_{i}"
        )(all_input)

        if i == embed_feat_idx:
            # CustomerID - embedding
            feat_int = layers.Lambda(
                lambda x: tf.cast(tf.squeeze(x, axis=-1), tf.int32),
                name="cust_id_int"
            )(feat_slice)
            emb = layers.Embedding(
                sg_n_rows, sg_dim,
                weights=[sg_matrix],
                trainable=False,
                name="CustomerID_embed",
            )(feat_int)
            feat_layers.append(emb)
        else:
            # Numeric feature - 2× causal Conv1D
            c1 = layers.Conv1D(
                2, 4, padding="causal", name=f"conv1_{conv_idx}"
            )(feat_slice)
            c2 = layers.Conv1D(
                2, 3, padding="causal", name=f"conv2_{conv_idx}"
            )(c1)
            feat_layers.append(c2)
            conv_idx += 1

    # Concatenate all processed features along the last axis
    if len(feat_layers) > 1:
        enc_input = layers.Concatenate(axis=-1, name="enc_input")(feat_layers)
    else:
        enc_input = feat_layers[0]

    # Encoder GRU 1
    _, h1 = layers.GRU(enc_units, dropout=dropout, return_state=True,
                        name="enc_gru1")(enc_input)

    # Dense bridge between encoder 1 and encoder 2
    h1_bridge = layers.Dense(enc_units, activation="tanh",
                               name="enc_bridge")(h1)

    # Encoder GRU 2 (reads same input, init from h1_bridge)
    _, h2 = layers.GRU(enc_units, dropout=dropout, return_state=True,
                        name="enc_gru2")(enc_input, initial_state=h1_bridge)

    # Dense bridge to decoder
    dec_init = layers.Dense(dec_units, activation="tanh",
                             name="dec_bridge")(h2)

    # Decoder GRU (reads same input, returns full sequence)
    dec_seq = layers.GRU(dec_units, dropout=dropout, return_sequences=True,
                          name="dec_gru")(enc_input, initial_state=dec_init)

    # Per-timestep projection
    dec_out = layers.TimeDistributed(layers.Dense(1), name="td_dense")(dec_seq)

    # Slice the last SEQ_OUT_LEN timesteps
    dec_out = layers.Lambda(
        lambda x: x[:, -seq_out_len:, :], name="slice_output"
    )(dec_out)

    model = Model(inputs=all_input, outputs=dec_out)
    model.compile(optimizer=optimizers.Adam(lr), loss="mse")
    return model

## 7. GBM Regression

 Prepares train/valid/test splits and DMatrix objects for the GBM regressor.
 Also exposes `buyer_train_idx` for the buyer-only training inside Section 9.
 Model training is performed in Section 9 (Pipeline Execution).

In [ ]:
# Build train / valid / test splits
train_times = list(range(MIN_TRAIN, MAX_TRAIN - TARGET_LEN + 1))
valid_times = [MIN_VALID - 1]
test_times  = [MIN_TEST  - 1]

cust_feats_all = cust_feats_all.merge(
    embed_wide, on="CustomerID", how="left"
)
subcat_col_names = [f"subcategoryNumberOfOrderIndicator_{i+1}"
                    for i in range(N_SUBCATEGORIES)]

# Row indices in the all-snapshot arrays
train_idx = cust_feats_all[cust_feats_all["DayID"].isin(train_times)].index.tolist()
valid_idx = cust_feats_all[cust_feats_all["DayID"].isin(valid_times)].index.tolist()
test_idx  = cust_feats_all[cust_feats_all["DayID"].isin(test_times) ].index.tolist()

BASE_EXCL   = {"CustomerID", "ItemID", "RowID", "DayID"}
feat_names  = [c for c in cust_feats_all.columns if c not in BASE_EXCL]

y_train_raw  = target_all.loc[train_idx, "CustomerTotalProfit"].values
outlier_mask = np.abs(y_train_raw) <= 1_000_000
clean_train_idx = [i for i, ok in zip(train_idx, outlier_mask) if ok]

# Buyer-only subset exposed for the stacking base GBM (not used here)
buyer_train_idx = [i for i, ok in zip(train_idx, outlier_mask & (y_train_raw > 0)) if ok]

y_train = target_all.loc[clean_train_idx, "CustomerTotalProfit"].values
y_valid = target_all.loc[valid_idx, "CustomerTotalProfit"].values
y_test  = target_all.loc[test_idx,  "CustomerTotalProfit"].values

valid_cust_gbm = cust_feats_all.loc[valid_idx, "CustomerID"].values
test_cust_gbm  = cust_feats_all.loc[test_idx,  "CustomerID"].values


def make_feat_matrix(row_idx, feat_names, cust_feats, subcat_sparse):
    dense  = cust_feats.loc[row_idx, feat_names].values.astype(np.float32)
    sparse = subcat_sparse[row_idx].toarray().astype(np.float32)
    mat    = np.hstack([dense, sparse])
    mat[~np.isfinite(mat)] = np.nan
    return mat


X_tr = make_feat_matrix(clean_train_idx, feat_names, cust_feats_all, subcat_sparse_all)
X_va = make_feat_matrix(valid_idx,       feat_names, cust_feats_all, subcat_sparse_all)
X_te = make_feat_matrix(test_idx,        feat_names, cust_feats_all, subcat_sparse_all)

dtrain = xgb.DMatrix(X_tr, label=y_train, missing=np.nan)
dvalid = xgb.DMatrix(X_va, label=y_valid, missing=np.nan)
dtest  = xgb.DMatrix(X_te, label=y_test,  missing=np.nan)

gbm_params = dict(
    objective="reg:squarederror",
    eta=ETA, max_depth=DEPTH,
    eval_metric="rmse",
    subsample=SUB, colsample_bytree=COL_SMP,
    booster="gbtree", seed=SEED,
)

 ## 8. Stage 1 — Purchase Propensity Classifier

 Prepares data for the XGBoost binary classifier (P(y > 0)).
 DMatrix objects and hyperparameters are set up here.
 Model training and Platt scaling calibration are performed in Section 9 (Pipeline Execution).

In [ ]:
# All available features (not just embedding subset) for the classifier
cls_feat_names = [c for c in cust_feats_all.columns if c not in BASE_EXCL]

y_cls_train = (target_all.loc[clean_train_idx, "CustomerTotalProfit"].values > 0).astype(float)
y_cls_valid = (y_valid > 0).astype(float)
y_cls_test  = (y_test  > 0).astype(float)

print(f"Class balance — valid: {y_cls_valid.mean():.4f}  test: {y_cls_test.mean():.4f}")

X_cls_tr = make_feat_matrix(clean_train_idx, cls_feat_names, cust_feats_all, subcat_sparse_all)
X_cls_va = make_feat_matrix(valid_idx,       cls_feat_names, cust_feats_all, subcat_sparse_all)
X_cls_te = make_feat_matrix(test_idx,        cls_feat_names, cust_feats_all, subcat_sparse_all)

dcls_train = xgb.DMatrix(X_cls_tr, label=y_cls_train, missing=np.nan)
dcls_valid = xgb.DMatrix(X_cls_va, label=y_cls_valid, missing=np.nan)
dcls_test  = xgb.DMatrix(X_cls_te, label=y_cls_test,  missing=np.nan)

cls_params = dict(
    objective="binary:logistic",
    eta=ETA, max_depth=DEPTH,
    eval_metric="auc",
    subsample=SUB, colsample_bytree=COL_SMP,
    booster="gbtree", seed=SEED,
)

 ## 9. Pipeline Execution — Model Training & Evaluation

 Trains all four models end-to-end, then evaluates them on the held-out test set.

 **Model 1:** Base XGBoost regressor (GBM)  
 **Model 2:** Base S2S encoder-decoder  
 **Model 3:** Stacking (XGBoost + S2S → XGBoost meta-learner)  
 **Model 4:** Two-stage (P(y>0) classifier × E[y|y>0] conditional regressor)  

 Final CLV prediction: `CLV_hat = P(y > 0) × E[y | y > 0]`

In [ ]:
import time

def _train_s2s_fresh(X_tr, Y_tr, X_val, Y_val):
    ckpt  = os.path.join(CACHE_PATH, '_tmp_s2s_timing.weights.h5')
    model = build_encoder_decoder(
        SEQ_IN_LEN, SEQ_OUT_LEN, NUM_SEQ_FEATS,
        EMBED_FEAT_IDX, sg_embedding_matrix,
        enc_units=ENC_UNITS, dec_units=DEC_UNITS, dropout=DROPOUT, lr=LR,
    )
    best_rmse, val_loss_vec = np.inf, []
    true_val = Y_val[:, :, 0].sum(axis=1)
    for epoch in range(N_EPOCHS):
        hist = model.fit(X_tr, Y_tr, epochs=1, batch_size=BATCH_SIZE,
                         validation_data=(X_val, Y_val), verbose=0)
        val_loss_vec.append(hist.history['val_loss'][0])
        pred = model.predict(X_val, verbose=0)[:, :, 0].sum(axis=1)
        rmse = np.sqrt(np.mean((pred - true_val) ** 2))
        if rmse < best_rmse:
            best_rmse = rmse
            model.save_weights(ckpt)
        patience_met = (len(val_loss_vec) > PATIENCE and
                        np.argmin(val_loss_vec) < len(val_loss_vec) - PATIENCE)
        explosion    = (epoch >= STOP_DELAY and
                        val_loss_vec[-1] >= min(val_loss_vec[:-1]) * STOP_FACTOR)
        if patience_met or explosion:
            break
    model.load_weights(ckpt)
    return model

def _xgb_fresh(params, dtrain, dvalid, nrounds, early, maximize=False):
    return xgb.train(params, dtrain, num_boost_round=nrounds,
                     evals=[(dvalid, 'valid')], early_stopping_rounds=early,
                     maximize=maximize, verbose_eval=False)

def _s2s_align(pred_arr, src_ids, tgt_ids):
    return (pd.Series(tgt_ids)
              .map(pd.Series(pred_arr, index=src_ids))
              .fillna(0).values)

def _stk_meta(X_dense, xgb_pred, s2s_pred, cls_pred=None):
    cols = [X_dense, xgb_pred[:, None], s2s_pred[:, None]]
    if cls_pred is not None:
        cols.append(cls_pred[:, None])
    mat = np.hstack(cols).astype(np.float32)
    mat[~np.isfinite(mat)] = np.nan
    return mat

M3_STK_PARAMS = dict(objective='reg:squarederror', eta=ETA, max_depth=DEPTH,
                     eval_metric='rmse', subsample=SUB, colsample_bytree=COL_SMP,
                     booster='gbtree', seed=SEED)

STK_PARAMS = dict(objective='reg:squarederror', eta=ETA, max_depth=4,
                  eval_metric='rmse', subsample=SUB, colsample_bytree=COL_SMP,
                  booster='gbtree', seed=SEED)

def compute_metrics(y_true, y_pred, label):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    order    = np.argsort(y_pred)[::-1]
    y_sorted = y_true[order]
    total    = y_sorted.sum()
    decile_lines = []
    for k in [10, 20, 30]:
        top_k    = max(1, int(np.ceil(len(y_true) * k / 100)))
        captured = y_sorted[:top_k].sum() / total if total > 0 else 0
        decile_lines.append(f'  Top-{k:2d}% lift:    {captured:.4f}  ({captured*100:.1f}% of spend)')
    print(f'\n{label}')
    print(f'  RMSE:           {rmse:.4f}')
    print(f'  MAE:            {mae:.4f}')
    print('\n'.join(decile_lines))

# Model 1: Base XGBoost
print('Training Model 1: Base XGBoost...')
t0 = time.perf_counter()
m1 = _xgb_fresh(gbm_params, dtrain, dvalid, NROUNDS, EARLY)
t_train_1 = time.perf_counter() - t0
t0 = time.perf_counter()
m1_test  = m1.predict(dtest)
t_infer_1 = time.perf_counter() - t0
print(f'  done  train={t_train_1:.1f}s  infer={t_infer_1:.3f}s')

# Model 2: Base S2S
print('Training Model 2: Base S2S...')
t0 = time.perf_counter()
m2 = _train_s2s_fresh(X_train, Y_train, X_valid, Y_valid)
t_train_2 = time.perf_counter() - t0
t0 = time.perf_counter()
m2_raw_test  = m2.predict(X_valid2, verbose=0)[:, :, 0].sum(axis=1)
t_infer_2 = time.perf_counter() - t0
m2_test  = _s2s_align(m2_raw_test,  valid2_cust_ids, test_cust_gbm)
print(f'  done  train={t_train_2:.1f}s  infer={t_infer_2:.3f}s')

# Model 3: Stacking (L1: XGBoost + S2S, L2: XGBoost on orig + L1 outputs)
print('Training Model 3: Stacking...')
t0 = time.perf_counter()
m3_l1_xgb    = _xgb_fresh(gbm_params, dtrain, dvalid, NROUNDS, EARLY)
m3_l1_s2s    = _train_s2s_fresh(X_train, Y_train, X_valid, Y_valid)
m3_l1_xgb_va = m3_l1_xgb.predict(dvalid)
m3_l1_xgb_te = m3_l1_xgb.predict(dtest)
m3_l1_s2s_va = _s2s_align(m3_l1_s2s.predict(X_valid,  verbose=0)[:, :, 0].sum(axis=1),
                            valid_cust_ids, valid_cust_gbm)
m3_l1_s2s_te = _s2s_align(m3_l1_s2s.predict(X_valid2, verbose=0)[:, :, 0].sum(axis=1),
                            valid2_cust_ids, test_cust_gbm)
m3_X_va = _stk_meta(X_va, m3_l1_xgb_va, m3_l1_s2s_va)
m3_X_te = _stk_meta(X_te, m3_l1_xgb_te, m3_l1_s2s_te)
rng = np.random.default_rng(SEED)
holdout_idx = rng.choice(len(y_valid), size=int(0.2 * len(y_valid)), replace=False)
train_idx   = np.setdiff1d(np.arange(len(y_valid)), holdout_idx)
dm3_train   = xgb.DMatrix(m3_X_va[train_idx],   label=y_valid[train_idx],   missing=np.nan)
dm3_holdout = xgb.DMatrix(m3_X_va[holdout_idx], label=y_valid[holdout_idx], missing=np.nan)
m3_stk = xgb.train(M3_STK_PARAMS, dm3_train,
                    num_boost_round=NROUNDS,
                    evals=[(dm3_holdout, 'holdout')],
                    early_stopping_rounds=EARLY,
                    verbose_eval=False)
t_train_3 = time.perf_counter() - t0
t0 = time.perf_counter()
m3_test = m3_stk.predict(xgb.DMatrix(m3_X_te, missing=np.nan))
t_infer_3 = time.perf_counter() - t0
print(f'  done  train={t_train_3:.1f}s  infer={t_infer_3:.3f}s')

# Model 4: Two-stage (Stage 1: classifier, Stage 2: stacking on buyers only)
print('Training Model 4: Two-stage...')
t0 = time.perf_counter()
m4_cls       = _xgb_fresh(cls_params, dcls_train, dcls_valid, NROUNDS, EARLY, maximize=True)
m4_prob_va   = m4_cls.predict(dcls_valid)
m4_prob_te   = m4_cls.predict(dcls_test)
m4_l1_xgb    = _xgb_fresh(gbm_params, dtrain, dvalid, NROUNDS, EARLY)
m4_l1_s2s    = _train_s2s_fresh(X_train, Y_train, X_valid, Y_valid)
m4_l1_xgb_va = m4_l1_xgb.predict(dvalid)
m4_l1_xgb_te = m4_l1_xgb.predict(dtest)
m4_l1_s2s_va = _s2s_align(m4_l1_s2s.predict(X_valid,  verbose=0)[:, :, 0].sum(axis=1),
                            valid_cust_ids, valid_cust_gbm)
m4_l1_s2s_te = _s2s_align(m4_l1_s2s.predict(X_valid2, verbose=0)[:, :, 0].sum(axis=1),
                            valid2_cust_ids, test_cust_gbm)
m4_X_va = _stk_meta(X_va, m4_l1_xgb_va, m4_l1_s2s_va, m4_prob_va)
m4_X_te = _stk_meta(X_te, m4_l1_xgb_te, m4_l1_s2s_te, m4_prob_te)
pos_mask = y_valid > 0
m4_stk  = xgb.train(STK_PARAMS, xgb.DMatrix(m4_X_va[pos_mask], label=y_valid[pos_mask], missing=np.nan),
                     num_boost_round=100, verbose_eval=False)
t_train_4 = time.perf_counter() - t0
t0 = time.perf_counter()
m4_test = m4_prob_te * m4_stk.predict(xgb.DMatrix(m4_X_te, missing=np.nan))
t_infer_4 = time.perf_counter() - t0
print(f'  done  train={t_train_4:.1f}s  infer={t_infer_4:.3f}s')

# Evaluation
compute_metrics(y_test,  m1_test,  '-- Model 1: Base XGBoost --')
compute_metrics(y_test,  m2_test,  '-- Model 2: Base S2S     --')
compute_metrics(y_test,  m3_test,  '-- Model 3: Stacking     --')
compute_metrics(y_test,  m4_test,  '-- Model 4: Two-stage    --')

# Runtime summary
print('\n' + '=' * 56)
print(f"{'Model':<30} {'Train (s)':>10} {'Infer (s)':>10}")
print('-' * 56)
for name, tt, ti in [
    ('1: Base XGBoost', t_train_1, t_infer_1),
    ('2: Base S2S',     t_train_2, t_infer_2),
    ('3: Stacking',     t_train_3, t_infer_3),
    ('4: Two-stage',    t_train_4, t_infer_4),
]:
    print(f'  {name:<28} {tt:>10.1f} {ti:>10.3f}')
print('=' * 56)
